In [4]:
#! pip install -U llama-index llama-index-llms-huggingface-api huggingface_hub

In [5]:
#! pip install llama-index-llms-openai-like -U -q

Test if we can connect to Qwen

In [6]:
from huggingface_hub import InferenceClient
import os

client = InferenceClient(
    api_key=os.environ["HF_TOKEN"],
    provider="auto",
)

response = client.chat.completions.create(
    model="Qwen/Qwen2.5-Coder-32B-Instruct",
    messages=[
        {
            "role": "user",
            "content": "What is 2 + 2?"
        }
    ],
    max_tokens=100,
    temperature=0.1,
)

print(response.choices[0].message.content)

2 + 2 equals 4.


In [7]:
import os
import chromadb

from huggingface_hub import InferenceClient
from llama_index.core.tools import QueryEngineTool
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.llms.openai_like import OpenAILike

In [8]:
from huggingface_hub import InferenceClient
import os

client = InferenceClient(
    api_key=os.environ["HF_TOKEN"],
    provider="auto",
)

response = client.chat.completions.create(
    model="Qwen/Qwen2.5-Coder-32B-Instruct",
    messages=[
        {"role": "user", "content": "What is 2 + 2?"}
    ],
)

print(response.choices[0].message.content)

2 + 2 equals 4.


In [9]:
HF_TOKEN = os.environ["HF_TOKEN"]

llm = OpenAILike(
    model="Qwen/Qwen2.5-Coder-32B-Instruct",
    api_base="https://router.huggingface.co/v1",
    api_key=HF_TOKEN,
    is_chat_model=True,
    temperature=0.1,
    max_tokens=512,
)

Chromadb

In [10]:
import chromadb

from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore

db = chromadb.PersistentClient(
    path="./alfred_chroma_db"
)

chroma_collection = db.get_or_create_collection(
    name="alfred"
)

vector_store = ChromaVectorStore(
    chroma_collection=chroma_collection
)

embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    embed_model=embed_model,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5037.25it/s]


Query Engine

In [11]:
query_engine = index.as_query_engine(
    llm=llm,
    similarity_top_k=3,
)

query_engine_tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="research_lookup",
    description=(
        "Search the research database for information about "
        "AI, artificial intelligence, the future of work, "
        "technology, and society."
    ),
    return_direct=False,
)

Calculator tools

In [12]:
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract b from a."""
    return a - b


def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


def divide(a: float, b: float) -> float:
    """Divide a by b."""
    if b == 0:
        raise ValueError("Cannot divide by zero.")

    return a / b

## Create Agents

Calculator agent

In [13]:
from llama_index.core.agent.workflow import (
    AgentWorkflow,
    ReActAgent,
)

calculator_agent = ReActAgent(
    name="calculator",
    description="Performs mathematical calculations.",
    system_prompt=(
        "You are a calculator assistant. "
        "Use your tools whenever arithmetic is required."
    ),
    tools=[
        add,
        subtract,
        multiply,
        divide,
    ],
    llm=llm,
)

Research Agent

In [14]:
query_agent = ReActAgent(
    name="info_lookup",
    description=(
        "Searches the research database for information "
        "about AI and society."
    ),
    system_prompt=(
        "You are a research assistant. "
        "Use the research_lookup tool whenever the answer "
        "can be found in the research database."
    ),
    tools=[
        query_engine_tool,
    ],
    llm=llm,
)

In [15]:
agent = AgentWorkflow(
    agents=[
        calculator_agent,
        query_agent,
    ],
    root_agent="calculator",
)

Run agent

In [16]:
from llama_index.core.agent.workflow import (
    AgentStream,
    ToolCallResult,
)

handler = agent.run(
    user_msg="What is (2 + 2) * 2?"
)

async for ev in handler.stream_events():

    if isinstance(ev, ToolCallResult):
        print("\n--- TOOL CALL ---")
        print("Tool:", ev.tool_name)
        print("Arguments:", ev.tool_kwargs)
        print("Result:", ev.tool_output)

    elif isinstance(ev, AgentStream):
        print(ev.delta, end="", flush=True)

response = await handler

print("\n\n--- FINAL RESPONSE ---")
print(response)

Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: add
Action Input: {"a": 2, "b": 2}
```
--- TOOL CALL ---
Tool: add
Arguments: {'a': 2, 'b': 2}
Result: 4
Thought: Now I need to multiply the result by 2.
Action: multiply
Action Input: {'a': 4, 'b': 2}
--- TOOL CALL ---
Tool: multiply
Arguments: {'a': 4, 'b': 2}
Result: 8
Thought: I can answer without using any more tools. I'll use the user's language to answer
Answer: (2 + 2) * 2 equals 8.

--- FINAL RESPONSE ---
(2 + 2) * 2 equals 8.
